# Practice Lab: Streaming & Real-Time Patterns (Lesson 7.2)

Module 7 · 8 exercises · SSE + WebSocket + streaming patterns

Exercises 1, 2, 5, 6, 7, 8 run locally (pure Python).


# Lesson 7.2: Streaming & Real-Time Patterns

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 5, 6, 7, 8 run **locally** (pure Python). Ex 3, 4 are architecture/design.


In [ ]:
import json, time, random, math
from concurrent.futures import ThreadPoolExecutor, as_completed


---
## Exercise 1 (Easy): SSE Protocol Basics

Build SSE messages with proper formatting.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

def format_sse(data, event=None, event_id=None):
    msg = ""
    if event_id is not None:
        msg += f"id: {event_id}\n"
    if event:
        msg += f"event: {event}\n"
    msg += f"data: {json.dumps(data)}\n\n"
    return msg

def format_heartbeat():
    return ": heartbeat\n\n"

tokens = ["The", "capital", "of", "India",
          "is", "New", "Delhi", "."]

print("SSE Stream Output:")
print("=" * 50)
total_bytes = 0
for i, token in enumerate(tokens):
    if i > 0 and i % 3 == 0:
        hb = format_heartbeat()
        print(hb, end="")
        total_bytes += len(hb.encode())

    msg = format_sse(
        data={"token": token, "index": i},
        event="token", event_id=i + 1)
    print(msg, end="")
    total_bytes += len(msg.encode())

done = "data: [DONE]\n\n"
print(done, end="")
total_bytes += len(done.encode())

print("=" * 50)
print(f"Tokens: {len(tokens)}")
print(f"Heartbeats: {len(tokens) // 3}")
print(f"Total bytes: {total_bytes}")
print(f"Avg bytes/token: {total_bytes / len(tokens):.0f}")

print("\nSSE Protocol Rules:")
for i, r in enumerate([
    "Content-Type: text/event-stream",
    "Cache-Control: no-cache",
    "Each message ends with \\n\\n",
    "Comments start with : (heartbeats)",
    "id: enables Last-Event-ID reconnection",
    "event: lets client filter by type",
    "data: [DONE] signals completion",
], 1):
    print(f"  {i}. {r}")


</details>


---
## Exercise 2 (Easy): Provider Streaming Comparison

Compare TTFB, speed, and event formats across 5 providers.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
providers = [
    {"name": "Groq (Llama 70B)", "ttfb_ms": 207,
     "tps": 540, "format": "OpenAI-compatible delta",
     "usage": "Final chunk only"},
    {"name": "OpenAI GPT-4o", "ttfb_ms": 500,
     "tps": 75, "format": "delta.content tokens",
     "usage": "Opt-in: stream_options"},
    {"name": "Google Gemini", "ttfb_ms": 1000,
     "tps": 50, "format": "Multi-sentence chunks",
     "usage": "Every chunk (cumulative)"},
    {"name": "Anthropic Claude", "ttfb_ms": 2000,
     "tps": 32, "format": "7-event lifecycle",
     "usage": "message_start + message_delta"},
    {"name": "DeepSeek V3", "ttfb_ms": 13000,
     "tps": 28, "format": "OpenAI-compatible",
     "usage": "Final chunk only"},
]

target = 500
print("Provider Streaming Comparison:")
print(f"{'Provider':<22} {'TTFB':<10} {'tok/s':<8} "
      f"{'500t time':<12} {'Format'}")
print("=" * 78)
for p in providers:
    gen = target / p["tps"]
    total = p["ttfb_ms"] / 1000 + gen
    print(f"{p['name']:<22} {p['ttfb_ms']:>6}ms "
          f"{p['tps']:>5}  {total:>7.1f}s     {p['format']}")

print("\n--- Event Format Samples ---")
print("\nOpenAI: choices[0].delta.content = single token")
print("Anthropic: message_start -> content_block_delta -> message_stop")
print("Gemini: multi-sentence chunks with cumulative metadata")

print("\nToken Usage Extraction:")
for p in providers:
    print(f"  {p['name']:<22} {p['usage']}")

fastest = min(providers, key=lambda p: p["ttfb_ms"])
slowest = max(providers, key=lambda p: p["ttfb_ms"])
ratio = slowest["ttfb_ms"] / fastest["ttfb_ms"]
print(f"\nTTFB range: {fastest['name']} ({fastest['ttfb_ms']}ms) "
      f"to {slowest['name']} ({slowest['ttfb_ms']}ms)")
print(f"Ratio: {ratio:.0f}x difference")


</details>


---
## Exercise 5 (Medium): Streaming Tool Calls

Accumulate partial JSON fragments from streaming tool calls.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

class ToolCallAccumulator:
    def __init__(self):
        self.calls = {}

    def process_delta(self, delta):
        if "tool_calls" not in delta:
            return
        for tc in delta["tool_calls"]:
            idx = tc["index"]
            if idx not in self.calls:
                self.calls[idx] = {"name": "", "args": ""}
            if "function" in tc:
                fn = tc["function"]
                if "name" in fn:
                    self.calls[idx]["name"] = fn["name"]
                if "arguments" in fn:
                    self.calls[idx]["args"] += fn["arguments"]

    def get_calls(self):
        results = []
        for idx in sorted(self.calls.keys()):
            c = self.calls[idx]
            try:
                args = json.loads(c["args"])
                results.append({"name": c["name"],
                                "arguments": args, "valid": True})
            except json.JSONDecodeError:
                results.append({"name": c["name"],
                                "arguments": c["args"], "valid": False})
        return results

deltas = [
    {"tool_calls": [{"index": 0, "function": {"name": "get_weather"}}]},
    {"tool_calls": [{"index": 0, "function": {"arguments": '{"'}}]},
    {"tool_calls": [{"index": 0, "function": {"arguments": 'city'}}]},
    {"tool_calls": [{"index": 0, "function": {"arguments": '": "'}}]},
    {"tool_calls": [{"index": 0, "function": {"arguments": 'Mumbai'}}]},
    {"tool_calls": [{"index": 0, "function": {"arguments": '"}'}}]},
    {"tool_calls": [{"index": 1, "function": {"name": "get_time"}}]},
    {"tool_calls": [{"index": 1, "function": {"arguments": '{"tz":'}}]},
    {"tool_calls": [{"index": 1, "function": {"arguments": '"Asia/Kolkata"}'}}]},
]

acc = ToolCallAccumulator()
print("Streaming Tool Call Accumulation:")
print("=" * 50)
for i, delta in enumerate(deltas):
    acc.process_delta(delta)
    for idx, call in acc.calls.items():
        print(f"  [{i+1:2d}] call[{idx}] "
              f"{call['name']:<14} args: {call['args']}")

results = acc.get_calls()
print(f"\nParsed Tool Calls ({len(results)}):")
for r in results:
    status = "OK" if r["valid"] else "INVALID"
    print(f"  {r['name']}({r['arguments']}) [{status}]")

print(f"\nParallel calls: {len(acc.calls)}")
print(f"All valid: {all(r['valid'] for r in results)}")
print(f"Fragments processed: {len(deltas)}")


</details>


---
## Exercise 6 (Challenge): Proxy Buffering Fix

Diagnose and fix reverse proxy buffering.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

layers = [
    {"layer": "nginx", "severity": "Critical",
     "default": "Buffers full response",
     "fix": "proxy_buffering off; proxy_http_version 1.1"},
    {"layer": "Cloudflare", "severity": "Critical",
     "default": "Buffers ~100KB before flush",
     "fix": "Enterprise: disable / Workers: ReadableStream"},
    {"layer": "AWS ALB/API GW", "severity": "Medium",
     "default": "REST API buffered until Nov 2025",
     "fix": "responseTransferMode: STREAM"},
    {"layer": "Cloud Run", "severity": "High",
     "default": "CPU throttled between requests",
     "fix": "--no-cpu-throttling --timeout 300"},
    {"layer": "Browser", "severity": "None",
     "default": "No buffering (SSE native)",
     "fix": "N/A"},
]

icons = {"Critical": "\U0001f534", "High": "\U0001f7e1",
         "Medium": "\U0001f7e0", "None": "\U0001f7e2"}

print("Reverse Proxy Buffering Diagnosis:")
print("=" * 55)
for l in layers:
    print(f"\n{icons[l['severity']]} {l['layer']} [{l['severity']}]")
    print(f"  Default: {l['default']}")
    print(f"  Fix: {l['fix']}")

def detect_buffering(times):
    if len(times) < 3:
        return {"buffered": False, "reason": "Too few tokens"}
    gaps = [times[i+1]-times[i] for i in range(len(times)-1)]
    first_gap = times[0]
    avg_gap = sum(gaps) / len(gaps)
    if first_gap > 2000 and avg_gap < 10:
        return {"buffered": True,
                "reason": f"First {first_gap}ms, then avg {avg_gap:.0f}ms"}
    if 20 < avg_gap < 200:
        return {"buffered": False,
                "reason": f"Consistent {avg_gap:.0f}ms gaps"}
    return {"buffered": "maybe",
            "reason": f"avg={avg_gap:.0f}ms"}

buffered = [5000, 5001, 5002, 5003, 5004, 5005]
streaming = [200, 250, 300, 350, 400, 450]

print("\n\nBuffering Detection:")
for name, times in [("Buffered", buffered), ("Streaming", streaming)]:
    print(f"  {name}: {detect_buffering(times)}")

print("\nDeployment Checklist:")
for i, item in enumerate([
    "StreamingResponse + text/event-stream",
    "Cache-Control: no-cache",
    "X-Accel-Buffering: no",
    "nginx: proxy_buffering off",
    "Cloud Run: --no-cpu-throttling",
    "Heartbeat every 15s",
    "Test TTFB < 500ms in production",
], 1):
    print(f"  [{i}] {item}")


</details>


---
## Exercise 7 (Challenge): Multi-Model Race

Race 3 providers, first-completed-wins.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import time, random
from concurrent.futures import ThreadPoolExecutor, as_completed

def simulate_provider(name, ttfb_ms, tps, fail_rate=0.0):
    if random.random() < fail_rate:
        raise Exception(f"{name} failed!")
    actual_ttfb = ttfb_ms * (0.8 + random.random() * 0.4)
    time.sleep(actual_ttfb / 1000)
    tokens = 50
    time.sleep(tokens / tps)
    return {"provider": name, "ttfb_ms": actual_ttfb,
            "total_ms": actual_ttfb + tokens/tps*1000}

providers = [
    ("Groq", 207, 540, 0.05),
    ("OpenAI", 500, 75, 0.10),
    ("Gemini", 1000, 50, 0.05),
]

print("Multi-Model Race:")
print("=" * 50)
with ThreadPoolExecutor(max_workers=3) as ex:
    futures = {ex.submit(simulate_provider, *p): p[0]
               for p in providers}
    winner = None
    for f in as_completed(futures):
        name = futures[f]
        try:
            r = f.result()
            if winner is None:
                winner = r
                print(f"  WINNER: {r['provider']} "
                      f"({r['total_ms']:.0f}ms)")
            else:
                print(f"  Slower: {r['provider']} "
                      f"({r['total_ms']:.0f}ms)")
        except Exception as e:
            print(f"  Failed: {name} ({e})")

strategies = [
    ("Single (Gemini)", 0.000165, 1500),
    ("Fallback chain", 0.000089, 400),
    ("Race (all 3)", 0.003004, 300),
]

print(f"\nCost Comparison (100K calls/month):")
print(f"{'Strategy':<25} {'$/call':<12} {'Monthly $':<10} {'Monthly INR'}")
print("-" * 60)
for name, cost, lat in strategies:
    monthly = cost * 100_000
    print(f"{name:<25} ${cost:<11.6f} ${monthly:<9.2f} "
          f"Rs {monthly*84:>8,.0f}")

print("\nRace: 2-3x cost for halved worst-case latency")
print("Fallback: best cost/reliability tradeoff")


</details>


---
## Exercise 8 (Challenge): India-Optimized Streaming

India-region latency + platform comparison.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
endpoints = [
    ("OpenAI US", 225, "GPT-4o", False),
    ("Anthropic US", 230, "Claude", False),
    ("Azure Central India", 21, "GPT-4o", True),
    ("Bedrock Mumbai", 18, "Claude/Llama", True),
    ("Vertex Mumbai", 20, "Gemini Flash", True),
    ("Sarvam AI", 12, "Sarvam-105B", True),
]

model_ttfb = 300
print("India Streaming Latency:")
print(f"{'Endpoint':<25} {'RTT':<8} {'Total TTFB':<12} {'INR billing'}")
print("=" * 60)
for name, rtt, models, inr in endpoints:
    total = model_ttfb + rtt
    flag = "\U0001f1ee\U0001f1f3" if inr else "\U0001f1fa\U0001f1f8"
    print(f"{flag} {name:<23} {rtt:>4}ms  {total:>6}ms     "
          f"{'Yes' if inr else 'No'}")

us_ttfb = model_ttfb + 225
india_ttfb = model_ttfb + 18
print(f"\nTTFB savings: {us_ttfb}ms -> {india_ttfb}ms "
      f"({us_ttfb - india_ttfb}ms / "
      f"{(us_ttfb-india_ttfb)/us_ttfb*100:.0f}%)")

platforms = [
    ("Web (SSE)", True, "350M+", "Full streaming"),
    ("WhatsApp", False, "500M+", "Complete response only"),
    ("Telegram", True, "120M+", "Bot API 9.3 streaming"),
    ("Mobile App", True, "Custom", "Native SSE + offline"),
]

print(f"\nIndian Platform Comparison:")
print(f"{'Platform':<16} {'Stream?':<10} {'Users':<10} {'Best For'}")
print("-" * 55)
for name, stream, users, best in platforms:
    print(f"{name:<16} {'Yes' if stream else 'NO':<10} "
          f"{users:<10} {best}")

print("\nMobile resilience features:")
for i, f in enumerate([
    "Last-Event-ID reconnection",
    "localStorage partial response cache",
    "Gzip SSE (~70% savings)",
    "Batch 3-5 tokens per event",
    "Works on 200Kbps rural 4G",
    "Heartbeat every 15s",
], 1):
    print(f"  {i}. {f}")

print("\nWhatsApp: 500M users, CANNOT stream")
print("Design: complete response on WA, streaming on Web/TG")


</details>


---
## Exercise 3 (Medium): Streaming Chat UI
Architecture/design exercise. See practice-lab-7_2.html.


In [ ]:
# Exercise 3: Streaming Chat UI
pass


---
## Exercise 4 (Medium): FastAPI → SSE → Frontend
Architecture/design exercise. See practice-lab-7_2.html.


In [ ]:
# Exercise 4: FastAPI → SSE → Frontend
pass
